# Create the ResNet18 LibTorch Artifact

Run this notebook in a Development workspace based on `nvcr.io/nvidia/pytorch:26.06-py3`. The PyTorch image already provides `torch` and `torchvision`. Do not install `torch` or `torchvision` in this notebook.

## 1. Bootstrap

In [ ]:
from pathlib import Path
import os
import sys


# The workspace container can run as a numeric uid that is not present in /etc/passwd.
# Set these before importing torch so PyTorch cache setup is stable.
os.environ.setdefault("USER", "workspace")
os.environ.setdefault("LOGNAME", "workspace")
os.environ.setdefault("TORCHINDUCTOR_CACHE_DIR", "/tmp/torchinductor-workspace")

ARTIFACT_PATH = Path("resnet18_libtorch/1/model.pt")

print("Python:", sys.executable)
print("Output:", ARTIFACT_PATH)

## 2. Check PyTorch, TorchVision, and CPU Runtime

In [ ]:
import torch
import torchvision


print("torch:", torch.__version__)
print("torch file:", torch.__file__)
print("torchvision:", torchvision.__version__)
print("torchvision file:", torchvision.__file__)

if "/workspace/.local" in torch.__file__:
    raise SystemExit("torch is loaded from /workspace/.local. Remove user-installed torch and restart the kernel.")
if "/workspace/.local" in torchvision.__file__:
    raise SystemExit("torchvision is loaded from /workspace/.local. Remove user-installed torchvision and restart the kernel.")

## 3. Load ResNet18

In [ ]:
from torchvision.models import ResNet18_Weights, resnet18


print("Loading ResNet18 weights...")
model = resnet18(weights=ResNet18_Weights.DEFAULT)
model.eval()
print("Model loaded")

## 4. Trace TorchScript

In [ ]:
example_input = torch.randn(1, 3, 224, 224)

with torch.inference_mode():
    logits = model(example_input)
    print("Smoke-test logits shape:", tuple(logits.shape))
    traced_model = torch.jit.trace(model, example_input, strict=False)

print("TorchScript trace created")

## 5. Save the Triton artifact

In [ ]:
ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)
traced_model.save(str(ARTIFACT_PATH))
print(f"Saved {ARTIFACT_PATH}")